#### Imports and Setup

In [2]:
import os
from dotenv import load_dotenv
from serpapi import GoogleSearch
#from IPython.display import Markdown
from newspaper import Article, Config
from newspaper.article import ArticleException
from concurrent.futures import ThreadPoolExecutor, as_completed
from selenium import webdriver
from selenium.webdriver import ChromeOptions
load_dotenv()

serp_api_key = os.environ.get('SERP_API_KEY')

In [1]:
import pandas as pd

df = pd.read_json('articles_with_text.json')

df

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text
0,dad963715e6a61d2481cc08d0aabf80a798d7e215bc36f...,1733081328483,en,ndtv.com,ndtv.com,1578960000000,Court Seeks Poll Body's Response On Plea For V...,https://www.ndtv.com/india-news/delhi-high-cou...,New Delhi: Delhi High Court on Monday directed...
1,a896d451b026e5a2c9e53dc2ea244d54e1d151f98f284c...,1733076288554,en,ndtv.com,ndtv.com,1578960000000,Indian-Origin British MP Inches Closer In Race...,https://www.ndtv.com/indians-abroad/indian-ori...,"Lisa Nandy, the Indian-origin British MP, and ..."
2,7bb1dced01bd427af43b311d3dba663d128695ed76163b...,1732968243987,en,ndtv.com,ndtv.com,1579910400000,"On National Voters' Day, PM Express Gratitude ...",https://www.ndtv.com/india-news/on-national-vo...,New Delhi: National Voters' Day or Rashtriya M...
3,b77aa012d1afa704fc1c449361adb64636ea303025a83f...,1732916929212,en,ndtv.com,ndtv.com,1580256000000,"""Bullet, Not Biryani"" For Anurag Thakur's Detr...",https://www.ndtv.com/india-news/delhi-assembly...,New Delhi: Karnataka Minister CT Ravi has back...
4,ffebb78904fec1139f8da6c90f5350146b3e4fda291d92...,1732804308510,en,ndtv.com,ndtv.com,1581292800000,"Behind Kejriwal’s Hanuman Chalisa Moment, A St...",https://www.ndtv.com/blog/behind-kejriwals-han...,"A few days before Delhi voted, a prominent new..."
...,...,...,...,...,...,...,...,...,...
286,a0e9be68ad6af1388d05b8f88b5fa9f4d3f71d0b09cfd9...,1766891832044,en,hindustantimes.com,hindustantimes.com,1766880000000,A bitter irony: Thackerays seek Muslim support...,https://www.hindustantimes.com/cities/mumbai-n...,"In 1992, the Sena was accused of being involve..."
287,fae5a44d732ff8902dfbe89b5420e974d2aed44981c9d6...,1766729986801,en,hindustantimes.com,hindustantimes.com,1766707200000,Myanmar to hold first general election in 5 yr...,https://www.hindustantimes.com/world-news/myan...,Myanmar will hold the first phase of a general...
288,e0a0b3e590fb6ab28fff1ed51351ccb047d8d433d1dc51...,1766629213711,en,hindustantimes.com,hindustantimes.com,1766620800000,"‘Not like Putin, Zelensky coming together’: BJ...",https://www.hindustantimes.com/cities/mumbai-n...,"Chief minister Devendra Fadnavis, deputy chief..."
289,84c7f7f103aba80019fa26e9b2986ea02d387128d97995...,1766542834097,en,hindustantimes.com,hindustantimes.com,1766534400000,NCP factions close to alliance for Pune civic ...,https://www.hindustantimes.com/cities/pune-new...,Leaders from both the NCP and NCP (SP) held a ...


In [3]:
def get_selenium_html(url):
    with webdriver.Chrome(options=options) as driver: 
        driver.get(url)
        article_html = driver.page_source
    return article_html

options = ChromeOptions()
options.page_load_strategy = 'eager'
options.add_argument("--headless=new")

config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
config.request_timeout = 20

#### Setup

#### Get SerpAPI Stories

In [4]:
def parse_url(url) -> str|None:
    try:
        article = Article(url, config=config)
        article.download()
        article.parse()

        text = " ".join(article.text.split()[:200])
        if text == '':
            raise LookupError(f"First Attempt: Unable to extact article text for {url}. Retrying...") # if it still fails, can't circumvent.

        return text
    
    except (ArticleException, LookupError) as e1:
        try:
            if str(e1).find('403') != -1 or isinstance(e1, LookupError):
                article = Article(url, config=config)
                article.download()
                article.parse()

                text = " ".join(article.text.split()[:200])
                if text == '':
                    raise LookupError(f"Second Attempt: Unable to extact article text for {url}. Retrying...") # if it still fails, can't circumvent.
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text
            
            else:
                print(e1)
                return
        except (ArticleException, LookupError) as e2:
            if str(e2).find('403') != -1 or isinstance(e2, LookupError):
                article = Article(url, config=config)
                article.download(input_html = get_selenium_html(url))
                article.parse()
                text = " ".join(article.text.split()[:200])
                if text == '':
                    text = None
                    print(f"Final Attempt: Unable to extact article text for {url}") # if it still fails, can't circumvent.
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text

            else:
                print(e2)
                return
    

def get_serp_stories(story_token:str):
    params = {
    "engine": "google_news",
    "gl": "in",
    "hl": "en",
    "story_token": story_token,
    "api_key": f"{serp_api_key}",
    #"no_cache": "true",
    "json_restrictor": "news_results[].stories[].{source.name, title, link, iso_date}, news_results[].{source.name, title, link, iso_date}"
    }

    search = GoogleSearch(params)
    #print(search.get_dict())

    stories = []

    for heading in search.get_dict()["news_results"]:
        if heading.get('title') == 'Posts on X':
            continue

        results = heading.get('stories', [heading])

        for story in results:
            stories.append({
                'title': story.get('title'),
                'outlet': story.get('source').get('name'),
                'url': story.get('link'),
                'publish_date': story.get('iso_date').partition('T')[0]
                })
            
    return stories

def parse_stories(stories:list[dict[str,str]]):
    for story in stories:
        url = story.get('url')
        text = parse_url(url)
        story['text'] = str(text)
    
    return stories

def parse_stories_parallel(stories, max_threads=5):
    
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        future_to_story = {
            executor.submit(parse_url, story['url']): story 
            for story in stories
        }
        
        for future in as_completed(future_to_story):
            story = future_to_story[future]
            try:
                text = future.result()
                story['text'] = text
                
            except Exception as e:
                print(f"Unexpected failure processing {story['url']}: {e}")
                story['text'] = None

    return stories

In [5]:
stories = get_serp_stories('CAAqNggKIjBDQklTSGpvSmMzUnZjbmt0TXpZd1NoRUtEd2pjay1YUkVCRWZfQURqZ0ZOQm1TZ0FQAQ')

stories, len(stories)

([{'title': 'Nitish Kumar: Veteran Bihar chief minister to step down for move to parliament',
   'outlet': 'BBC',
   'url': 'https://www.bbc.com/news/articles/cr455pr24wpo',
   'publish_date': '2026-03-05'},
  {'title': 'Video | BJP To Get Another CM: Allies On Eggshells?',
   'outlet': 'NDTV',
   'url': 'https://www.ndtv.com/video/bjp-to-get-another-cm-allies-on-eggshells-1068465',
   'publish_date': '2026-03-05'},
  {'title': 'With Amit Shah by his side, Bihar CM Nitish Kumar files nomination for Rajya Sabha',
   'outlet': 'The Times of India',
   'url': 'https://timesofindia.indiatimes.com/india/new-govt-will-have-my-full-cooperation-nitish-kumar-bids-adieu-to-bihar-politics-eyes-rajya-sabha-seat/articleshow/129071574.cms',
   'publish_date': '2026-03-05'},
  {'title': 'Rajya Sabha election LIVE: Bihar CM Nitish Kumar files nomination papers for Rajya Sabha polls',
   'outlet': 'The Hindu',
   'url': 'https://www.thehindu.com/news/national/rajya-sabha-election-2026-congress-bjp-mva-

In [6]:
stories = parse_stories_parallel(stories)

stories

Final Attempt: Unable to extact article text for https://www.ndtv.com/video/bjp-to-get-another-cm-allies-on-eggshells-1068465
Final Attempt: Unable to extact article text for https://www.telegraphindia.com/india/rajya-sabha-route-for-nitish-kumar-kharges-prediction-returns-bjp-weighs-bihar-power-shift/cid/2149835
Retry succeeded for https://www.wionews.com/india-news/nitish-kumar-era-ends-in-bihar-as-cm-moves-to-rajya-sabha-hectic-parleys-underway-for-new-leadership-1772688620111!
Retry succeeded for https://www.india.com/news/india/nitish-kumar-janata-dal-united-jdu-rajya-sabha-amit-shah-salary-bungalow-flat-hostel-tejashwi-yadav-rjd-bjp-upper-house-narendra-modi-8331183/!
Final Attempt: Unable to extact article text for https://www.ndtv.com/india-news/nitish-kumar-rajya-sabha-theres-been-a-desire-in-my-heart-nitish-kumar-to-move-to-rajya-sabha-11170762
Final Attempt: Unable to extact article text for https://www.ndtv.com/india-news/rajya-sabha-bihar-nitish-kumar-may-quit-as-bihar-chi

[{'title': 'Nitish Kumar: Veteran Bihar chief minister to step down for move to parliament',
  'outlet': 'BBC',
  'url': 'https://www.bbc.com/news/articles/cr455pr24wpo',
  'publish_date': '2026-03-05',
  'text': 'Nitish Kumar, the chief minister of India\'s eastern state of Bihar, has said he will step down from his post to become a member of the federal parliament. Kumar, 75, said in a post on X on Thursday that the new government in the state "will have his full cooperation and guidance". His decision paves the way for a new chief minister, who can be from Kumar\'s Janata Dal (United) party or his coalition partner Bharatiya Janata Party (BJP). This decision marks a pause for Kumar\'s political career in the state where he was chief minister for most parts of the last two decades. He is one of Bihar\'s most influential political figures. Several of his party colleagues and alliance partners say his move to vacate the top post has been on the cards for a while because of his deterior

In [ ]:
texts = [story.get('text') for story in stories if story.get('text') is not None]

texts

In [ ]:
from twikit import Client # pyright: ignore[reportMissingImports]

USERNAME = 'NA'
PASSWORD = 'NA'

client = Client('en-IN')

await client.login(
        auth_info_1=USERNAME,
        password=PASSWORD,
        cookies_file='cookies.json'
    )

user = await client.get_user_by_screen_name('elonmusk')
print(user.name)

In [7]:
import os
import pandas as pd
import requests
import time
from dotenv import load_dotenv
load_dotenv()

x_api_key = os.environ.get('X_API_KEY')

def get_display_name(username):
    time.sleep(0.05)
    url = "https://api.twitterapi.io/twitter/user/info"
    headers = {
        "X-API-Key": x_api_key
    }
    params = {
        "userName": username
    }
    
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    
    data = response.json()

    try:
        display_name = data["data"]["name"]
        print(display_name)
        return display_name
    except:
        print("failed!")
        return pd.NA

In [ ]:
df = pd.read_json('india_July_21.json')
df = df[df['party'] != 'To Be Added']

df

In [ ]:
df_test = pd.DataFrame(df)
df_test['display_name'] = df_test['screen_name'].apply(get_display_name)

df_test

In [7]:
df_test.to_json('nivaduck_with_display_names.json')
df_test.to_csv('nivaduck_with_display_names.csv')

In [1]:
import spacy
import pandas as pd
from thefuzz import fuzz, process
from NewsSentiment import TargetSentimentClassifier
newsmtsc_classifier = TargetSentimentClassifier()

nlp = spacy.load('en_core_web_trf')

/Users/rupsharray/Documents/Ashoka/Research/IndiaMediaLens/IndiaMediaLens/.iml_venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
df_corpus = pd.read_json('nivaduck_with_display_names.json')
df_corpus = df_corpus.dropna(subset='display_name')

corpus = set(df_corpus['display_name'])
corpus.remove('')

corpus

{'Ncp Mira Bhaindar City District',
 'Gopi Nath Pandey',
 'Akhilesh Patidar Iyc',
 'Shiv yadav',
 'Afaque Ahmad Khan',
 'Inc Bidar',
 'Adv. Shivpal Ubnare',
 'Akhilesh Chaudhary',
 'Amarjeet Singh Raja (Modi Ka Pariwar)',
 'Pankaj Thakur (BJP) (मोदी का परिवार)',
 'Nanabhau Mohod',
 'Pratap Baburao Sarnaik',
 'param Desai',
 'Prabhu ADMK-Say NoTo Drugs & Dmk',
 'Sanjay Khambayate - संजय खंबायते',
 'Rajajinagar AAP',
 'Narender Nath Ex. MLA',
 'V.H.Choudhary',
 'Prafulla Kr. Mahanta',
 'NSUI Live',
 'hariom Prabhakar',
 'Smriti Irani Office',
 'chokidaar satya prakash dhakad',
 'ABHISHEK BHARGAD🇮🇳',
 'Asha Rathore.ITCell n social media district conven',
 'Samadhan Sarvankar',
 'Beninson cijo - Say No To Drugs & DMK',
 'Gopal Vaghela',
 '@Jaydeepmakwana09',
 'Ravindra V Gaikwad',
 'Sachchidanand Upasane FAN',
 'Dhandev Thakur',
 'Hadi Pranjal',
 'Rajkumar Mev',
 'Adv.Tej Pratap Singh & Associates',
 'RAMRAO MAHALE BJP Ex MAL sausar',
 'Ashok chavan @INC',
 'Tahir Sayeed',
 'Punitt Upadhya

In [12]:
import random

random.seed(42)
text_samples = random.sample(texts, 10)

text_samples

["NEW DELHI/PATNA: Less than four months after Nitish Kumar, along with PM Modi, led NDA to a landslide win in the Bihar assembly elections in Nov and took oath as chief minister for a record 10th time, 'sushashan babu', who turned 75 recently, is set to move to the Rajya Sabha - paving the way for BJP to have its own CM in the state for the first time, and extending its jurisdictional imprint across almost all of north and central India. His son Nishant is set to enter politics by joining the government; while speculation abounds that he might be named deputy CM, there is as yet little clarity on his position or portfolio. Big Bihar Surprise: CM Nitish Kumar To Contest Rajya Sabha Polls, BJP Set to Rule Bihar People in the know said Nitish will file his nomination for the Rajya Sabha in Patna Thursday in the presence of key NDA members, including Union home minister Amit Shah. Four other NDA nominees, including BJP president Nitin Nabin and Union minister Ram Nath Thakur, and Rashtriy

In [15]:
def get_data_from_doc(doc, corpus):
    entities = []
    datapoints = []

    for ent in doc.ents:
        if ent.label_ not in {"PERSON", "ORG"}:
            continue
        
        match = process.extractOne(ent.text, corpus, scorer=fuzz.token_set_ratio, score_cutoff=90)
        if not match:
            continue
        
        sentence = ent.sent
        left = doc.text[sentence.start_char : ent.start_char]
        entity_str = ent.text
        right= doc.text[ent.end_char : sentence.end_char]
        
        entities.append(entity_str)
        datapoints.append((left, entity_str, right))
    
    return entities, datapoints

def batch_classify(data, data_index_map, stories, classifier, batch_size=32):
    if not data:
        return

    sentiments = classifier.infer(targets=data, batch_size=batch_size)
    
    for global_idx, result in enumerate(sentiments):
        story_idx = data_index_map[global_idx]
        story = stories[story_idx]
        
        if 'sentiments' not in story:
            story['sentiments'] = []
        story['sentiments'].append(result)

def batch_newsmtsc_annotate(stories, classifier, batch_size=32):
    plaintexts = [story.get('text') for story in stories if story.get('text') is not None]
    indices = [i for i, story in enumerate(stories) if story.get('text') is not None]

    docs = nlp.pipe(plaintexts, batch_size=batch_size)

    data = []
    data_index_map = {}

    for i, doc in enumerate(docs):
        entities, datapoints = get_data_from_doc(doc, corpus)
        
        story = stories[indices[i]]
        story['entities'] = entities
        
        for dp in datapoints:
            global_idx = len(data)
            
            data.append(dp)
            data_index_map[global_idx] = indices[i]
    
    batch_classify(data, data_index_map, stories, classifier, batch_size)


def batch_annotate_stories_optimized(plaintexts, corpus, classifier, batch_size=32):
    docs = nlp.pipe(plaintexts, batch_size=batch_size)
    
    for doc in docs:
        data = []
        
        for ent in doc.ents:
            if ent.label_ not in {"PERSON", "ORG"}:
                continue
        

            match = process.extractOne(ent.text, corpus, scorer=fuzz.token_set_ratio, score_cutoff=90)
            if not match:
                #print("match failed")
                continue
            
            #print(ent)
            #print("match success")
            sentence = ent.sent

            #print(sentence.text)
            
            left_context = doc.text[sentence.start_char : ent.start_char]
            target_text = ent.text
            right_context = doc.text[ent.end_char : sentence.end_char]
            
            data.append((left_context, target_text, right_context))

        if data:

            for datapt in data: print(datapt)
            sentiments = classifier.infer(targets=data, batch_size=batch_size)
            for i, result in enumerate(sentiments):
                print("Sentiment: ", i, result[0], flush=True)
        
        print(flush=True)

In [16]:
plaintexts = texts
#['Nirmala Sitharaman was defending the Centre\'s position by commenting on opposition\'s allegations that Centre is slashing funds for centrally sponsored schemes.']

batch_annotate_stories_optimized(plaintexts, corpus, newsmtsc_classifier)

('', 'Nitish Kumar', ", the chief minister of India's eastern state of Bihar, has said he will step down from his post to become a member of the federal parliament.")
('', 'Kumar', ', 75, said in a post on X on Thursday that the new government in the state "will have his full cooperation and guidance".')
('Kumar, 75, said in a post on ', 'X', ' on Thursday that the new government in the state "will have his full cooperation and guidance".')
('His decision paves the way for a new chief minister, who can be from ', 'Kumar', "'s Janata Dal (United) party or his coalition partner Bharatiya Janata Party (BJP).")
("His decision paves the way for a new chief minister, who can be from Kumar's ", 'Janata Dal', ' (United) party or his coalition partner Bharatiya Janata Party (BJP).')
("His decision paves the way for a new chief minister, who can be from Kumar's Janata Dal (", 'United', ') party or his coalition partner Bharatiya Janata Party (BJP).')
("His decision paves the way for a new chief 

Processing batches: 100%|██████████| 1/1 [00:00<00:00,  1.22batch/s]

Sentiment:  0 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.6359891295433044}
Sentiment:  1 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.5986576676368713}
Sentiment:  2 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.8662275075912476}
Sentiment:  3 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.6166033148765564}
Sentiment:  4 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7121310830116272}
Sentiment:  5 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7309185862541199}
Sentiment:  6 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7572458982467651}
Sentiment:  7 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7969009876251221}
Sentiment:  8 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.4570715129375458}
Sentiment:  9 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.9077084064483643}
Sentiment:  10 {'class_id': 0, 'class_label': 'negative', 'class_prob': 0.9164807796478271}
Sentime

('Bihar chief minister ', 'Nitish Kumar', ' on Thursday filed his nomination for the upcoming Rajya Sabha elections.')
('Bihar chief minister Nitish Kumar on Thursday filed his nomination for the upcoming ', 'Rajya Sabha', ' elections.')
('He was accompanied by Union Home Minister ', 'Amit Shah', ' during the filing of his nomination papers.')
('This comes after ', 'Nitish', ' announced his decision to step down from the state’s top job and head to the Rajya Sabha.')
('', 'Nitish Kumar’s', ' Rajya Sabha Decision Sparks Protest, Anger Inside JD(U) Ranks')
('Nitish Kumar’s ', 'Rajya Sabha', ' Decision Sparks Protest, Anger Inside JD(U) Ranks')
('In a social media post on ', 'X', ', Nitish said he had long harboured a desire to become a member of both Houses of the Bihar Legislative Assembly as well as both Houses of Parliament.')
('In a social media post on X, ', 'Nitish', ' said he had long harboured a desire to become a member of both Houses of the Bihar Legislative Assembly as well as

Processing batches: 100%|██████████| 1/1 [00:00<00:00,  1.69batch/s]

...

Processing batches: 100%|██████████| 1/1 [00:00<00:00,  1.58batch/s]

Sentiment:  0 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.9638561606407166}
Sentiment:  1 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.5406945943832397}
Sentiment:  2 {'class_id': 0, 'class_label': 'negative', 'class_prob': 0.7710917592048645}
Sentiment:  3 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.9531424641609192}
Sentiment:  4 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.7262991070747375}
Sentiment:  5 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.9153911471366882}
Sentiment:  6 {'class_id': 0, 'class_label': 'negative', 'class_prob': 0.9751395583152771}
Sentiment:  7 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.503463864326477}
Sentiment:  8 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.895326554775238}
Sentiment:  9 {'class_id': 0, 'class_label': 'negative', 'class_prob': 0.6995581984519958}



('With Bihar Chief Minister ', 'Nitish Kumar', ' himself confirming on social media on Thursday (March 5, 2026) that he “seeks to become a member of the Rajya Sabha in the elections being held this time”, all eyes are now set on who would be the next Chief Minister of the State and what role Mr. Kumar’s son Nishant Kumar would play in the coming days in the State’s politics.')
('With Bihar Chief Minister Nitish Kumar himself confirming on social media on Thursday (March 5, 2026) that he “seeks to become a member of the Rajya Sabha in the elections being held this time”, all eyes are now set on who would be the next Chief Minister of the State and what role Mr. ', 'Kumar', '’s son Nishant Kumar would play in the coming days in the State’s politics.')
('With Bihar Chief Minister Nitish Kumar himself confirming on social media on Thursday (March 5, 2026) that he “seeks to become a member of the Rajya Sabha in the elections being held this time”, all eyes are now set on who would be the ne

Processing batches: 100%|██████████| 1/1 [00:00<00:00,  1.39batch/s]

Sentiment:  0 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.649990975856781}
Sentiment:  1 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.5250837802886963}
Sentiment:  2 {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.5377026200294495}
Sentiment:  3 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.979094922542572}
Sentiment:  4 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7657076716423035}
Sentiment:  5 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.604972779750824}
Sentiment:  6 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.6866859793663025}
Sentiment:  7 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.7848789691925049}
Sentiment:  8 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.5613539814949036}
Sentiment:  9 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.9516453146934509}
Sentiment:  10 {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.9456043243408203}

